In [1]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Parameters
n_entities = 20  # Number of companies/individuals
n_time = 5       # Number of time periods (20 * 5 = 100 observations)

# Create panel data structure
entity_ids = np.repeat(range(1, n_entities + 1), n_time)
time_periods = np.tile(range(2019, 2019 + n_time), n_entities)

# Create entity-specific fixed effects (unobserved heterogeneity)
# Each entity has its own "baseline" level
entity_effects = np.repeat(np.random.normal(50, 15, n_entities), n_time)

# Create time-specific fixed effects (common shocks)
time_effects = np.tile([0, 5, 3, -2, 4], n_entities)

# Generate independent variables
np.random.seed(42)
advertising = np.random.uniform(10, 100, 100)
price = np.random.uniform(20, 80, 100)
competition = np.random.randint(1, 10, 100)

# Generate dependent variable with fixed effects
# Sales = entity_effect + time_effect + β1*advertising + β2*price + β3*competition + error
sales = (
    entity_effects +           # Entity fixed effect
    time_effects +             # Time fixed effect
    0.8 * advertising +        # True coefficient
    -0.5 * price +             # True coefficient
    -2 * competition +         # True coefficient
    np.random.normal(0, 10, 100)  # Random error
)

# Create DataFrame
panel_data = pd.DataFrame({
    'entity_id': entity_ids,
    'year': time_periods,
    'sales': sales,
    'advertising': advertising,
    'price': price,
    'competition': competition
})

# Add some entity characteristics (time-invariant)
entity_chars = pd.DataFrame({
    'entity_id': range(1, n_entities + 1),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n_entities),
    'company_size': np.random.choice(['Small', 'Medium', 'Large'], n_entities)
})

panel_data = panel_data.merge(entity_chars, on='entity_id')

# Sort by entity and time
panel_data = panel_data.sort_values(['entity_id', 'year']).reset_index(drop=True)

print("Panel Data Sample (N=100):")
print(panel_data.head(15))
print(f"\nShape: {panel_data.shape}")
print(f"\nEntities: {panel_data['entity_id'].nunique()}")
print(f"Time periods: {panel_data['year'].nunique()}")
print(f"\nSummary Statistics:")
print(panel_data.describe())


Panel Data Sample (N=100):
    entity_id  year       sales  advertising      price  competition region  \
0           1  2019   66.973062    43.708611  21.885751            8  North   
1           1  2020  103.218904    95.564288  58.184625            4  North   
2           1  2021  103.267793    75.879455  38.861359            1  North   
3           1  2022   46.493877    63.879264  50.514241            8  North   
4           1  2023   45.120980    24.041678  74.453988            4  North   
5           2  2019   32.569429    24.039507  34.957534            6  South   
6           2  2020   38.900554    15.227525  44.622975            8  South   
7           2  2021   79.111824    87.955853  65.333068            4  South   
8           2  2022   70.597156    64.100351  33.727890            3  South   
9           2  2023   67.685703    73.726532  24.618795            9  South   
10          3  2019   36.849774    11.852604  37.385087            3  South   
11          3  2020  113.

In [3]:
panel_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   entity_id     100 non-null    int64  
 1   year          100 non-null    int64  
 2   sales         100 non-null    float64
 3   advertising   100 non-null    float64
 4   price         100 non-null    float64
 5   competition   100 non-null    int32  
 6   region        100 non-null    object 
 7   company_size  100 non-null    object 
dtypes: float64(3), int32(1), int64(2), object(2)
memory usage: 6.0+ KB


In [4]:
panel_data.nunique()

entity_id        20
year              5
sales           100
advertising     100
price           100
competition       9
region            4
company_size      3
dtype: int64

In [ ]:
fsize=pd.get_dummies(panel_data['company_size'], prefix='s', drop_first=True)

df=pd.concat([panel_data, fsize], axis=1)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   entity_id     100 non-null    int64  
 1   year          100 non-null    int64  
 2   sales         100 non-null    float64
 3   advertising   100 non-null    float64
 4   price         100 non-null    float64
 5   competition   100 non-null    int32  
 6   region        100 non-null    object 
 7   company_size  100 non-null    object 
 8   s_Medium      100 non-null    bool   
 9   s_Small       100 non-null    bool   
dtypes: bool(2), float64(3), int32(1), int64(2), object(2)
memory usage: 6.2+ KB


In [12]:
from statsmodels.formula.api import ols

m1=ols('sales~price+s_Medium+s_Small', data=df).fit()
m1.params
m1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  sales   R-squared:                       0.108
Model:                            OLS   Adj. R-squared:                  0.080
Method:                 Least Squares   F-statistic:                     3.855
Date:                Tue, 17 Feb 2026   Prob (F-statistic):             0.0119
Time:                        20:46:03   Log-Likelihood:                -473.55
No. Observations:                 100   AIC:                             955.1
Df Residuals:                      96   BIC:                             965.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
====================================================================================
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           88.6789     10.747      8.251      0.000      67.345     110.012
s_Medium[T.True]    -4.7843      8.525     -0.561      0.576     -21.707      12.138
s_Small[T.True]     -6.7562      8.388     -0.805      0.423     -23.406       9.894
price               -0.5326      0.161     -3.305      0.001      -0.852      -0.213
==============================================================================
Omnibus:                        0.290   Durbin-Watson:                   1.558
Prob(Omnibus):                  0.865   Jarque-Bera (JB):                0.166
Skew:                           0.100   Prob(JB):                        0.920
Kurtosis:                       3.000   Cond. No.                         261.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [11]:
df0=df[['price','s_Medium','s_Small']].copy()
df1=df0.assign(sales_hat=m1.predict(df0))
df1.head(10)

,price,s_Medium,s_Small,sales_hat
0,21.885751,False,True,70.266698
1,58.184625,False,True,50.934487
2,38.861359,False,True,61.225756
3,50.514241,False,True,55.019613
4,74.453988,False,True,42.269680
5,34.957534,False,True,63.304871
6,44.622975,False,True,58.157209
7,65.333068,False,True,47.127339
8,33.727890,False,True,63.959760
9,24.618795,False,True,68.811122


# t test


In [13]:
import pandas as pd
import numpy as np
from scipy import stats

np.random.seed(42)

# Simulate A/B test data
n_control = 500    # Group A (control - old design)
n_treatment = 500  # Group B (treatment - new design)

# Spending is often right-skewed (lognormal distribution is more realistic)
control_spending = np.random.lognormal(mean=3.5, sigma=1.2, size=n_control)    # Old design
treatment_spending = np.random.lognormal(mean=3.7, sigma=1.2, size=n_treatment) # New design

# Create DataFrame
ab_data = pd.DataFrame({
    'user_id': range(1, n_control + n_treatment + 1),
    'group': ['control'] * n_control + ['treatment'] * n_treatment,
    'spending': np.concatenate([control_spending, treatment_spending]),
    'age': np.random.randint(18, 65, n_control + n_treatment),
    'days_active': np.random.randint(1, 30, n_control + n_treatment)
})

# Add some zero spenders (users who didn't buy)
zero_spenders = np.random.choice(ab_data.index, size=200, replace=False)
ab_data.loc[zero_spenders, 'spending'] = 0

print("Sample Data:")
print(ab_data.head(10))
print(f"\nShape: {ab_data.shape}")


Sample Data:
   user_id    group    spending  age  days_active
0        1  control   60.102833   61           18
1        2  control   28.052643   57            3
2        3  control   72.040340   28           13
3        4  control  205.950496   20           11
4        5  control   25.003503   23           17
5        6  control   25.003996   26           22
6        7  control  220.314347   23           13
7        8  control   83.172908   26            6
8        9  control   18.852221   56           17
9       10  control   63.502041   48           27

Shape: (1000, 5)


$$df = \frac{\left(\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}\right)^2}{\frac{\left(\frac{s_1^2}{n_1}\right)^2}{n_1 - 1} + \frac{\left(\frac{s_2^2}{n_2}\right)^2}{n_2 - 1}}$$

Where:

$s_1^2, s_2^2$ = sample variances of group 1 and 2
$n_1, n_2$ = sample sizes of group 1 and 2

$$SE_{Welch} = \sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}} \quad \text{(each group keeps its own variance)}$$

$$SE_{pooled} = \sqrt{s_p^2\left(\frac{1}{n_1} + \frac{1}{n_2}\right)} \quad \text{(one shared variance estimate)}$$

In [16]:
s_c=ab_data.loc[ab_data['group']=='control', 'spending'].to_numpy()
s_t=ab_data.loc[ab_data['group']=='treatment', 'spending'].to_numpy()

se=np.sqrt(s_c.var(ddof=1)/len(s_c)+s_t.var(ddof=1)/len(s_t))
t_stats=np.abs((s_c.mean()-s_t.mean())/se)

from scipy.stats import t
var_c = s_c.var(ddof=1) / len(s_c)
var_t = s_t.var(ddof=1) / len(s_t)

df_welch = (var_c + var_t)**2 / (var_c**2/(len(s_c)-1) + var_t**2/(len(s_t)-1))
p_value=2*(1-t.cdf(t_stats, df=df_welch))
print(p_value)


0.32720779059576977
